# SpatioAI Phase 4: Spatio-Temporal Graph Neural Network (GNN) Tracking

This notebook demonstrates the end-to-end Spatio-Temporal Graph Neural Network for meteorological event tracking, comparing the classical Hungarian baseline tracker with the learned GNN association model.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from src.gnn.dataset import generate_synthetic_event_sequence, split_events_by_track
from src.gnn.graph_builder import SpatioTemporalGraphBuilder, GraphBuildConfig
from src.gnn.inference import GNNPredictor
from src.gnn.evaluation import compute_classification_metrics, evaluate_track_reconstruction, compare_baseline_vs_gnn
from scripts.evaluate_gnn import evaluate_baseline_tracker

print(f"PyTorch Version: {torch.__version__}")

## 1. Generate Multi-System Synthetic Extreme Weather Trajectories

In [ ]:
events, track_labels = generate_synthetic_event_sequence(
    num_storms=6,
    timesteps_per_storm=8,
    time_step_hours=6.0,
    seed=42
)
print(f"Total Events: {len(events)}, Total Tracks: {len(set(track_labels.values()))}")
df_events = pd.DataFrame(events)
df_events.head()

## 2. Evaluate Baseline Hungarian Tracker vs Spatio-Temporal GNN

In [ ]:
# Split into train and test tracks
_, _, test_events = split_events_by_track(events, track_labels, train_ratio=0.6, val_ratio=0.2, seed=42)
print(f"Test Events: {len(test_events)}")

# Evaluate Baseline Tracker
baseline_metrics, baseline_tracks = evaluate_baseline_tracker(test_events, track_labels)

# Evaluate GNN Tracker
predictor = GNNPredictor(checkpoint_path="models/spatiotemporal_gnn_best.pt", association_threshold=0.50)
gnn_results = predictor.predict_events(test_events)

gt_tracks_dict = {}
for e in sorted(test_events, key=lambda x: pd.to_datetime(x["timestamp"])):
    trk_id = track_labels.get(e["event_id"], "UNKNOWN")
    gt_tracks_dict.setdefault(trk_id, []).append(e["event_id"])
gt_tracks = list(gt_tracks_dict.values())

gnn_tracking_metrics = evaluate_track_reconstruction(gnn_results["predicted_tracks"], gt_tracks)

edge_pairs = gnn_results["edge_pairs"]
edge_probs = gnn_results["probabilities"]
y_true = [1.0 if (track_labels.get(src) and track_labels.get(dst) and track_labels.get(src) == track_labels.get(dst)) else 0.0 for src, dst in edge_pairs]
gnn_cls_metrics = compute_classification_metrics(y_true, edge_probs, threshold=0.5)

gnn_metrics = {**gnn_cls_metrics, **gnn_tracking_metrics}
df_comparison = compare_baseline_vs_gnn(baseline_metrics, gnn_metrics)
df_comparison

## 3. Visualize Track Predictions and Association Graphs

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Plot Baseline Tracks
ax1.set_title("Baseline Classical Tracker", fontsize=14, fontweight="bold")
for trk in baseline_tracks:
    evts = [e for e in test_events if e["event_id"] in trk]
    lats = [e["centroid_lat"] for e in evts]
    lons = [e["centroid_lon"] for e in evts]
    ax1.plot(lons, lats, marker="o", markersize=6, linewidth=2)
ax1.set_xlabel("Longitude (°E)")
ax1.set_ylabel("Latitude (°N)")
ax1.grid(True, alpha=0.3)

# Plot GNN Reconstructed Tracks
ax2.set_title("Spatio-Temporal GNN Tracker", fontsize=14, fontweight="bold")
for trk in gnn_results["predicted_tracks"]:
    evts = [e for e in test_events if e["event_id"] in trk]
    lats = [e["centroid_lat"] for e in evts]
    lons = [e["centroid_lon"] for e in evts]
    ax2.plot(lons, lats, marker="s", markersize=6, linewidth=2)
ax2.set_xlabel("Longitude (°E)")
ax2.set_ylabel("Latitude (°N)")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()